# Visualize Colab CPU smoke + one FPV clip

Synthetic `demo-fixture` frames are **not** patio FPV. GCS clips under `runid00x` are owner video. This notebook does not train, invoke Cosmos, or move the robot.

In [ ]:
import json
from pathlib import Path
from IPython.display import Video, display, Markdown
from PIL import Image
import matplotlib.pyplot as plt

SMOKE = Path("/tmp/cpu-smoke")
GCS_SMOKE = "gs://caferoomba/artifacts/colab-cpu-smoke/cpu-smoke"
GCS_CLIP = "gs://caferoomba/media/data/raw/fpv/runid001/clips/part_01.mp4"
LOCAL_CLIP = Path("/tmp/fpv_runid001_part01.mp4")

if not (SMOKE / "manifest.json").is_file():
    !gcloud storage cp -r {GCS_SMOKE} /tmp/cpu-smoke-from-gcs
    SMOKE = Path("/tmp/cpu-smoke-from-gcs")

manifest = json.loads((SMOKE / "manifest.json").read_text())
display(Markdown(f"**synthetic={manifest.get('is_synthetic')}** evidence={manifest.get('evidence_state')} command=`{manifest.get('command')}`"))
print(json.dumps({k: manifest.get(k) for k in ('kind', 'is_synthetic', 'evidence_state')}, indent=2))
print("extras keys", sorted((manifest.get("extras") or {}).keys()))

In [ ]:
frame_root = SMOKE / "frames"
runs = sorted(p for p in frame_root.iterdir() if p.is_dir()) if frame_root.is_dir() else []
n = min(6, len(runs))
fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
if n == 1:
    axes = [axes]
for ax, run in zip(axes, runs[:n]):
    img = Image.open(next(run.glob("frame_*.png")))
    ax.imshow(img)
    ax.set_title(run.name, fontsize=8)
    ax.axis("off")
fig.suptitle("SYNTHETIC fixture frames (not FPV)")
plt.show()

In [ ]:
if not LOCAL_CLIP.is_file():
    !gcloud storage cp {GCS_CLIP} {LOCAL_CLIP}
display(Markdown("**Owner FPV clip** `runid001/clips/part_01.mp4` (not a training label, not Cosmos)."))
display(Video(str(LOCAL_CLIP), embed=True, width=480))